# IIT-09 — Le capstone : intégré pour toute coupe, trivial en deux pas

**Marche 3 (capstone) de l'escalier `iit_lean`**. Les deux premières marches — le verdict de coupe,
puis l'outil parité et le système broadcast — sont assemblées ici en un seul énoncé : **la critique
d'Aaronson est formellement tenable**. Un système peut être intégré au sens du proxy *pour toute
coupe* (théorème B) et n'avoir qu'une dynamique à deux pas (théorème C) — les deux sur le **même**
objet.

Ce capstone **consomme** au lieu de redéfinir : le vocabulaire du verdict vient de la marche 1
([IIT-07]), l'outil parité et les orbites de la marche 2 ([IIT-08]), et — hors lake — les
**mesures Φ** du carnet Python [IIT-06-L], dont ce carnet est « la moitié formelle ». Le grade
honnête du lake est porté par le carnet lui-même (§6) : le proxy mesure la décomposition
déterministe de l'image, rien d'autre.


## Navigation
**Série** : [IIT](./README.md) | **Partie** : Escalier Lean (marche 3/3)
**Précédent** : [IIT-8 : la parité, de bout en bout](./IIT-08-Parite-De-Bout-En-Bout-Lean.ipynb) | **Suivant** : --

[IIT-07]: ./IIT-07-Integration-Par-Coupes-Lean.ipynb
[IIT-08]: ./IIT-08-Parite-De-Bout-En-Bout-Lean.ipynb
[IIT-06-L]: ../IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb

## 0. Ce que ce carnet démontre

1. Le **défaut cardinal** du proxy — `|imA| · |imB| − |im f|`, la quantité que le lake renvoie
   en Phase 2 — se **mesure par énumération** dès que `n` est concret : 2 pour le broadcast,
   0 pour l'identité (§3).
2. Le **mécanisme** du théorème B se compte : chaque motif de coupe est réalisé séparément
   (4 états sur 8), aucun couple incompatible n'a de témoin (0 sur 8) — la non-indépendance,
   comptée au lieu d'affirmée (§3).
3. Le couple **B + C** tient sur le même système avec les seuls axiomes standard (§4) :
   intégration universelle, dynamique triviale.
4. Mis en regard des **mesures Φ** de [IIT-06-L] — la famille W qui sature Φ_DM à 50 %, ring3
   à Φ = 1,875 — le parallèle est structurant mais **borné** : le lake formalise un proxy, le
   carnet Python mesure un Φ ; les deux ne se comparent pas terme à terme (§5-§6).

In [1]:
import IIT.Integration
open IIT

-- Le contrat du capstone : le couple B/C et le verdict, consommes sans redefinir
#check @integrated
#check @decomposable
#check @broadcast_integrated
#check @broadcast_two_cycle
#check @broadcast_trivial_dynamics

import IIT.Integration
open IIT

-- Le contrat du capstone : le couple B/C et le verdict, consommes sans redefinir
#check @integrated
──────▶  @integrated : {n : Nat} → System n → Prop
#check @decomposable
──────▶  @decomposable : {n : Nat} → System n → Prop
#check @broadcast_integrated
──────▶  @broadcast_integrated : ∀ {n : Nat}, integrated broadcast
#check @broadcast_two_cycle
──────▶  @broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#check @broadcast_trivial_dynamics
──────▶  @broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
--% env 0

Raw input:
{"cmd": "import IIT.Integration\nopen IIT\n\n-- Le contrat du capstone : le couple B/C et le verdict, consommes sans redefinir\n#check @integrated\n#check @decomposable\n#check @broadcast_integrated\n#check @broadcast_two_cycle\n#check @broadcast_trivial_dynamics"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "@integrated : {n : Nat} → System n → Prop"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "@decomposable : {n : Nat} → System n → Prop"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "@broadcast_integrated : ∀ {n : Nat}, integrated broadcast"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "@broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"}],
 "env": 0}

Lecture du contrat : `integrated f` dit qu'**aucune** coupe de `f` n'est indépendante
(`¬ decomposable f`) ; le théorème B (`broadcast_integrated`) l'établit pour le
broadcast-parité ; le théorème C (`broadcast_two_cycle`, alias figé
`broadcast_trivial_dynamics` de l'issue #18599) dit que toute orbite du même système atteint
un point fixe en deux pas. Ce sont les énoncés 17-19 de l'inventaire du lake — le capstone
les assemble au lieu de les re-démontrer.

## 1. Le verdict, sans le redéfinir

Rappel de la marche 1, en trois définitions : une coupe partage les `n` éléments en deux côtés
non vides ; les motifs `projT`/`projF` gardent les bits d'un côté et mettent l'autre à `false` ;
la coupe est **indépendante** si l'image du système est exactement le **produit** des images de
ses deux parties.

In [2]:
-- Le vocabulaire du verdict (marche 1)
#check @Cut
#check @projT
#check @projF
#check @cutIndep

-- Le vocabulaire du verdict (marche 1)
#check @Cut
──────▶  Cut : Nat → Type
#check @projT
──────▶  @projT : {n : Nat} → Cut n → State n → State n
#check @projF
──────▶  @projF : {n : Nat} → Cut n → State n → State n
#check @cutIndep
──────▶  @cutIndep : {n : Nat} → Cut n → System n → Prop
--% env 1

Raw input:
{"cmd": "-- Le vocabulaire du verdict (marche 1)\n#check @Cut\n#check @projT\n#check @projF\n#check @cutIndep", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "Cut : Nat → Type"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "@projT : {n : Nat} → Cut n → State n → State n"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "@projF : {n : Nat} → Cut n → State n → State n"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "@cutIndep : {n : Nat} → Cut n → System n → Prop"}],
 "env": 1}

L'équivalence qui fait tenir la suite : `cutIndep c f` demande que **tout couple de motifs
réalisables séparément soit réalisable ensemble** — c'est la lecture « produit d'images » de
l'indépendance. Alors :

- `decomposable f` = il **existe** une coupe indépendante ;
- `integrated f` = **aucune** coupe ne l'est.

Le théorème A (marche 1) dit que l'identité est du côté décomposable ; le théorème B que le
broadcast est du côté intégré. Le capstone mesure **les deux** dans le même formalisme.

## 2. Le système, relu en une mesure

Le broadcast-parité (marche 2) : chaque bit de l'état suivant vaut la parité de l'état courant.
Une seule mesure de rappel — le reste (lemmes, orbites) vit dans l'[IIT-08].

In [3]:
-- Rappel marche 2 : un pas de broadcast sur un temoin
def sOneHot : State 3 := fun j => decide (j.val = 0)
def bitsOf {n : Nat} (s : State n) : List Bool := (List.finRange n).map s

#eval bitsOf sOneHot
#eval bitsOf (broadcast sOneHot)

-- Rappel marche 2 : un pas de broadcast sur un temoin
def sOneHot : State 3 := fun j => decide (j.val = 0)
def bitsOf {n : Nat} (s : State n) : List Bool := (List.finRange n).map s

#eval bitsOf sOneHot
─────▶  [true, false, false]
#eval bitsOf (broadcast sOneHot)
─────▶  [true, true, true]
--% env 2

Raw input:
{"cmd": "-- Rappel marche 2 : un pas de broadcast sur un temoin\ndef sOneHot : State 3 := fun j => decide (j.val = 0)\ndef bitsOf {n : Nat} (s : State n) : List Bool := (List.finRange n).map s\n\n#eval bitsOf sOneHot\n#eval bitsOf (broadcast sOneHot)", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[true, false, false]"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "[true, true, true]"}],
 "env": 2}

Lecture : l'état `(true, false, false)` (parité impaire) passe à `(true, true, true)` —
l'information globale est diffusée partout. L'image entière du broadcast ne contient que des
états **constants** : c'est la moitié « trivial » du couple, que la mesure suivante rend
exacte.

## 3. Le défaut cardinal, mesuré par énumération

L'outil du capstone : énumérer les `2^n` états, projeter les images sur les deux côtés de la
coupe, dédoublonner. Le lake renvoie la cardinalité quantitative `|imA| · |imB| − |im f|` en
Phase 2 ; **un carnet n'attend pas une phase** — elle se mesure.

In [4]:
-- Enumeration des etats et images decalees en listes de bits
def stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

def imageBits (n : Nat) (f : System n) : List (List Bool) :=
  ((allStates n).map (fun s => bitsOf (f s))).eraseDups

#eval (imageBits 3 broadcast).length
#eval imageBits 3 broadcast

-- Enumeration des etats et images decalees en listes de bits
def stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

def imageBits (n : Nat) (f : System n) : List (List Bool) :=
  ((allStates n).map (fun s => bitsOf (f s))).eraseDups

#eval (imageBits 3 broadcast).length
─────▶  2
#eval imageBits 3 broadcast
─────▶  [[false, false, false], [true, true, true]]
--% env 3

Raw input:
{"cmd": "-- Enumeration des etats et images decalees en listes de bits\ndef stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1\ndef allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)\n\ndef imageBits (n : Nat) (f : System n) : List (List Bool) :=\n  ((allStates n).map (fun s => bitsOf (f s))).eraseDups\n\n#eval (imageBits 3 broadcast).length\n#eval imageBits 3 broadcast", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 5},
   "data": "[[false, false, false], [true, true, true]]"}],
 "env": 3}

Mesure : l'image du broadcast en taille 3 est **exactement** `{(false, false, false),
(true, true, true)}` — deux états constants, rien d'autre. Sur ces 8 états de départ, tout ce
qui sort est uniforme.

In [5]:
-- Images partielles sur une coupe concrete : cote A = {indice 0}, cote B = {1, 2}
def cutA : Cut 3 := ⟨fun i => i.val == 0, ⟨0, by decide⟩, ⟨1, by decide⟩⟩

def partImageT (n : Nat) (c : Cut n) (f : System n) : List (List Bool) :=
  ((allStates n).map (fun s => bitsOf (projT c (f s)))).eraseDups
def partImageF (n : Nat) (c : Cut n) (f : System n) : List (List Bool) :=
  ((allStates n).map (fun s => bitsOf (projF c (f s)))).eraseDups

-- (|imA|, |imB|, |imA| * |imB|, |im f|) pour le broadcast
#eval ((partImageT 3 cutA broadcast).length,
       (partImageF 3 cutA broadcast).length,
       (partImageT 3 cutA broadcast).length * (partImageF 3 cutA broadcast).length,
       (imageBits 3 broadcast).length)

-- Images partielles sur une coupe concrete : cote A = {indice 0}, cote B = {1, 2}
def cutA : Cut 3 := ⟨fun i => i.val == 0, ⟨0, by decide⟩, ⟨1, by decide⟩⟩

def partImageT (n : Nat) (c : Cut n) (f : System n) : List (List Bool) :=
  ((allStates n).map (fun s => bitsOf (projT c (f s)))).eraseDups
def partImageF (n : Nat) (c : Cut n) (f : System n) : List (List Bool) :=
  ((allStates n).map (fun s => bitsOf (projF c (f s)))).eraseDups

-- (|imA|, |imB|, |imA| * |imB|, |im f|) pour le broadcast
#eval ((partImageT 3 cutA broadcast).length,
─────▶  (2, 2, 4, 2)
       (partImageF 3 cutA broadcast).length,
       (partImageT 3 cutA broadcast).length * (partImageF 3 cutA broadcast).length,
       (imageBits 3 broadcast).length)
--% env 4

Raw input:
{"cmd": "-- Images partielles sur une coupe concrete : cote A = {indice 0}, cote B = {1, 2}\ndef cutA : Cut 3 := \u27e8fun i => i.val == 0, \u27e80, by decide\u27e9, \u27e81, by decide\u27e9\u27e9\n\ndef partImageT (n : Nat) (c : Cut n) (f : System n) : List (List Bool) :=\n  ((allStates n).map (fun s => bitsOf (projT c (f s)))).eraseDups\ndef partImageF (n : Nat) (c : Cut n) (f : System n) : List (List Bool) :=\n  ((allStates n).map (fun s => bitsOf (projF c (f s)))).eraseDups\n\n-- (|imA|, |imB|, |imA| * |imB|, |im f|) pour le broadcast\n#eval ((partImageT 3 cutA broadcast).length,\n       (partImageF 3 cutA broadcast).length,\n       (partImageT 3 cutA broadcast).length * (partImageF 3 cutA broadcast).length,\n       (imageBits 3 broadcast).length)", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "(2, 2, 4, 2)"}],
 "env": 4}

Mesure : `(|imA|, |imB|, produit, |im f|) = (2, 2, 4, 2)`. Les images partielles ont
chacune **deux** motifs (la parité vrai/faux projetée sur un côté), leur produit en promet
**quatre** — l'image n'en réalise que **deux**. Le défaut cardinal vaut `4 − 2 = 2` : la coupe
n'est pas indépendante, et l'écart est **un nombre**, pas une adjectif.

In [6]:
-- La meme mesure sur l'identite : le theoreme A, rendu par les nombres
#eval ((partImageT 3 cutA (fun s => s)).length,
       (partImageF 3 cutA (fun s => s)).length,
       (partImageT 3 cutA (fun s => s)).length * (partImageF 3 cutA (fun s => s)).length,
       (imageBits 3 (fun s => s)).length)

#check @cutIndep_of_sides_independent
#check @product_decomposable

-- La meme mesure sur l'identite : le theoreme A, rendu par les nombres
#eval ((partImageT 3 cutA (fun s => s)).length,
─────▶  (2, 4, 8, 8)
       (partImageF 3 cutA (fun s => s)).length,
       (partImageT 3 cutA (fun s => s)).length * (partImageF 3 cutA (fun s => s)).length,
       (imageBits 3 (fun s => s)).length)

#check @cutIndep_of_sides_independent
──────▶  @cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      cutIndep c f
#check @product_decomposable
──────▶  @product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      decomposable f
--% env 5

Raw input:
{"cmd": "-- La meme mesure sur l'identite : le theoreme A, rendu par les nombres\n#eval ((partImageT 3 cutA (fun s => s)).length,\n       (partImageF 3 cutA (fun s => s)).length,\n       (partImageT 3 cutA (fun s => s)).length * (partImageF 3 cutA (fun s => s)).length,\n       (imageBits 3 (fun s => s)).length)\n\n#check @cutIndep_of_sides_independent\n#check @product_decomposable", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "(2, 4, 8, 8)"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      cutIndep c f"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      decomposable f"}],
 "env": 5}

Mesure : pour l'identité, `(2, 4, 8, 8)` — le produit des images partielles **égale**
l'image (`8 = 8`), défaut `0` : la coupe est indépendante, le système est décomposable. C'est
le théorème A (`cutIndep_of_sides_independent`, alias figé `product_decomposable`) rendu par
les nombres.

| système | (\|imA\|, \|imB\|, produit, \|im f\|) | défaut | verdict |
|---|---|---|---|
| identité | (2, 4, 8, **8**) | **0** | décomposable (théorème A) |
| broadcast | (2, 2, 4, **2**) | **2** | intégré (théorème B) |

In [7]:
-- Le mecanisme de B, compte : temoins par paire de motifs
def pairWitnessCount (c : Cut 3) (px : Bool) (py : Bool) : Nat :=
  ((allStates 3).filter (fun z =>
     (bitsOf (projT c (broadcast z)) == bitsOf (projT c (broadcast (stateOfNat 3 (if px then 1 else 0)))))
     && (bitsOf (projF c (broadcast z)) == bitsOf (projF c (broadcast (stateOfNat 3 (if py then 1 else 0))))))).length

-- (T,T) (T,F) (F,T) (F,F) : parites (cote A, cote B) des deux motifs demandes
#eval (pairWitnessCount cutA true true, pairWitnessCount cutA true false,
       pairWitnessCount cutA false true, pairWitnessCount cutA false false)

-- Le mecanisme de B, compte : temoins par paire de motifs
def pairWitnessCount (c : Cut 3) (px : Bool) (py : Bool) : Nat :=
  ((allStates 3).filter (fun z =>
     (bitsOf (projT c (broadcast z)) == bitsOf (projT c (broadcast (stateOfNat 3 (if px then 1 else 0)))))
     && (bitsOf (projF c (broadcast z)) == bitsOf (projF c (broadcast (stateOfNat 3 (if py then 1 else 0))))))).length

-- (T,T) (T,F) (F,T) (F,F) : parites (cote A, cote B) des deux motifs demandes
#eval (pairWitnessCount cutA true true, pairWitnessCount cutA true false,
─────▶  (4, 0, 0, 4)
       pairWitnessCount cutA false true, pairWitnessCount cutA false false)
--% env 6

Raw input:
{"cmd": "-- Le mecanisme de B, compte : temoins par paire de motifs\ndef pairWitnessCount (c : Cut 3) (px : Bool) (py : Bool) : Nat :=\n  ((allStates 3).filter (fun z =>\n     (bitsOf (projT c (broadcast z)) == bitsOf (projT c (broadcast (stateOfNat 3 (if px then 1 else 0)))))\n     && (bitsOf (projF c (broadcast z)) == bitsOf (projF c (broadcast (stateOfNat 3 (if py then 1 else 0))))))).length\n\n-- (T,T) (T,F) (F,T) (F,F) : parites (cote A, cote B) des deux motifs demandes\n#eval (pairWitnessCount cutA true true, pairWitnessCount cutA true false,\n       pairWitnessCount cutA false true, pairWitnessCount cutA false false)", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "(4, 0, 0, 4)"}],
 "env": 6}

Mesure : `(4, 0, 0, 4)`. Chaque motif est réalisé séparément — 4 états sur 8 diffusent
`true` sur le côté A, 4 autres diffusent `false` sur le côté B — mais **aucun état ne réalise
les deux à la fois** : les cases hors diagonale sont **nulles**. C'est exactement ce que
`cutIndep` demande et que le broadcast refuse : un couple de motifs séparément réalisables,
jamais réalisable ensemble. Le théorème B dit que ceci arrive pour **toute** coupe ; la
mesure suivante le vérifie sur une deuxième.

In [8]:
-- Deuxieme coupe : cote A = {0, 1}, cote B = {2}
def cutB : Cut 3 := ⟨fun i => i.val < 2, ⟨0, by decide⟩, ⟨2, by decide⟩⟩

#eval ((partImageT 3 cutB broadcast).length,
       (partImageF 3 cutB broadcast).length,
       (partImageT 3 cutB broadcast).length * (partImageF 3 cutB broadcast).length,
       (imageBits 3 broadcast).length)
#eval (pairWitnessCount cutB true true, pairWitnessCount cutB true false,
       pairWitnessCount cutB false true, pairWitnessCount cutB false false)

-- Deuxieme coupe : cote A = {0, 1}, cote B = {2}
def cutB : Cut 3 := ⟨fun i => i.val < 2, ⟨0, by decide⟩, ⟨2, by decide⟩⟩

#eval ((partImageT 3 cutB broadcast).length,
─────▶  (2, 2, 4, 2)
       (partImageF 3 cutB broadcast).length,
       (partImageT 3 cutB broadcast).length * (partImageF 3 cutB broadcast).length,
       (imageBits 3 broadcast).length)
#eval (pairWitnessCount cutB true true, pairWitnessCount cutB true false,
─────▶  (4, 0, 0, 4)
       pairWitnessCount cutB false true, pairWitnessCount cutB false false)
--% env 7

Raw input:
{"cmd": "-- Deuxieme coupe : cote A = {0, 1}, cote B = {2}\ndef cutB : Cut 3 := \u27e8fun i => i.val < 2, \u27e80, by decide\u27e9, \u27e82, by decide\u27e9\u27e9\n\n#eval ((partImageT 3 cutB broadcast).length,\n       (partImageF 3 cutB broadcast).length,\n       (partImageT 3 cutB broadcast).length * (partImageF 3 cutB broadcast).length,\n       (imageBits 3 broadcast).length)\n#eval (pairWitnessCount cutB true true, pairWitnessCount cutB true false,\n       pairWitnessCount cutB false true, pairWitnessCount cutB false false)", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "(2, 2, 4, 2)"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "(4, 0, 0, 4)"}],
 "env": 7}

Mesure : `(2, 2, 4, 2)` et `(4, 0, 0, 4)` — **identiques à la première coupe**. Le défaut
cardinal du broadcast (2) et le motif des témoins (diagonale 4, hors-diagonale 0) ne dépendent
pas de la coupe : trois coupes existent en taille 3, les deux mesurées les montrent toutes
identiques, et le théorème B clôt les trois — puis toutes les tailles.

## 4. Le couple en théorèmes

Les mesures ci-dessus sont finies (une taille, des coupes). Les théorèmes du lake les
remplacent par des énoncés universels — c'est le passage qui fait la valeur du formel.

In [9]:
-- Le couple B/C et ses dependances
#check @broadcast_integrated
#print axioms broadcast_integrated
#check @broadcast_two_cycle
#print axioms broadcast_two_cycle
#check @broadcast_trivial_dynamics
#print axioms broadcast_trivial_dynamics

-- Le couple B/C et ses dependances
#check @broadcast_integrated
──────▶  @broadcast_integrated : ∀ {n : Nat}, integrated broadcast
#print axioms broadcast_integrated
──────▶  'IIT.broadcast_integrated' depends on axioms: [propext]
#check @broadcast_two_cycle
──────▶  @broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#print axioms broadcast_two_cycle
──────▶  'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]
#check @broadcast_trivial_dynamics
──────▶  @broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#print axioms broadcast_trivial_dynamics
──────▶  'IIT.broadcast_trivial_dynamics' depends on axioms: [propext, Quot.sound]
--% env 8

Raw input:
{"cmd": "-- Le couple B/C et ses dependances\n#check @broadcast_integrated\n#print axioms broadcast_integrated\n#check @broadcast_two_cycle\n#print axioms broadcast_two_cycle\n#check @broadcast_trivial_dynamics\n#print axioms broadcast_trivial_dynamics", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "@broadcast_integrated : ∀ {n : Nat}, integrated broadcast"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "'IIT.broadcast_integrated' depends on axioms: [propext]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "@broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "'IIT.broadcast_trivial_dynamics' depends on axioms: [propext, Quot.sound]"}],
 "env": 8}

Mesure des dépendances : le théorème B repose sur `propext` **seul** ; le théorème C
ajoute `Quot.sound` (qui porte l'extensionnalité des fonctions de sa conclusion point à point).
Aucun axiome ajouté, aucun `sorry` — le couple qui porte la critique d'Aaronson est aussi fort
que le langage le permet, et il tient **sur le même système** :

- **B** : `integrated broadcast` — intégration universelle au sens du proxy ;
- **C** : toute orbite atteint un point fixe en deux pas — dynamique triviale.

C'est la critique, formellement tenable : *un système peut maximiser le proxy d'intégration
et n'avoir qu'une dynamique à deux pas.*

## 5. La mise en regard des mesures Φ de l'[IIT-06-L]

Le carnet Python [IIT-06-L] (sur main) mesure l'objet de la critique **numériquement**, sur
des instances réelles. Ses sorties committées, citées telles quelles :

- **Φ_DM de l'identité** (n = 8, p = 1009) : `PHI final : 0.000000 bits` — inversible et
  totalement décomposable : Φ nul pour l'objet le plus « riche » en comportement local ;
- **la famille W** (Vandermonde) : `PHI = (n/2)·log₂(p)`, rapport au contenu total
  `50.0 %` constant de n = 4 à n = 16 (`19.96/39.91` à `39.91/79.83`) — la mesure est
  **saturée** par un objet de description triviale, « franchement indigeste » ;
- **PyPhi 3.0** : `ring3` (XOR) → `Phi = 1.87500` (la référence de la série), `line6`
  (majorité) → `Phi = 0.02778`.

Le parallèle, colonne par colonne :

| | côté Python ([IIT-06-L]) | côté Lean (ce carnet) |
|---|---|---|
| objet qui « mord » | la famille W (Vandermonde) | le broadcast-parité |
| ce qui est mesuré | Φ_DM : minimum de rangs normalisés | le défaut cardinal par coupe |
| le verdict extrême | Φ = 50 % du contenu total, saturé | intégré pour **toute** coupe (théorème B) |
| la trivialité | description de quelques lignes | point fixe en deux pas (théorème C) |
| le contre-point | l'identité : Φ_DM = 0 | l'identité : défaut 0, décomposable (théorème A) |

Deux formalisations de la **même structure de critique** : un objet trivial à décrire sature
la mesure, pendant que l'objet « riche » est déclaré décomposable.

## 6. Le grade honnête — ce que le proxy ne capture pas

Porté par le carnet lui-même, comme l'exige la note de conception du lake :

- le proxy mesure la **décomposition déterministe de l'image** : deux systèmes de même image
  ont même verdict, quelle que soit leur dynamique intermédiaire ;
- il ne capture **ni** l'information effective sous perturbation, **ni** la partition minimale
  au sens MIP, **ni** une valeur numérique de Φ ;
- le carnet Python **mesure** des Φ (deux jams : Φ_DM matriciel, Φ 3.0 de PyPhi) ; le lake
  **formalise un proxy**. Les deux ne se comparent pas terme à terme — le tableau du §5 aligne
  des **correspondances structurelles**, pas des égalités de nombres ;
- la source de la critique est un **billet de blog (2014)**, pas une publication formelle :
  le lake **crée** l'objet formel (le couple B/C sur le broadcast), il n'en est pas une
  transcription.

Ce que le couple B/C établit n'est donc pas « IIT est fausse » — c'est : *le critère
d'intégration par coupe, pris littéralement, récompense un objet à dynamique nulle*. La
réponse de Tononi (changer de définition, mesurer sous perturbation) vit dans l'échange cité
par l'[IIT-06-L] §1, et reste un débat philosophique — cité comme tel.

## 7. Conclusion de l'escalier

Les trois marches sont assemblées :

1. **marche 1** ([IIT-07]) : le verdict — coupe, motifs, indépendance ; l'identité est
   décomposable (théorème A) ;
2. **marche 2** ([IIT-08]) : l'outil — la parité et ses lemmes, le broadcast et ses orbites ;
3. **marche 3** (ce carnet) : l'assemblage — le défaut cardinal mesuré (2 pour le broadcast,
   0 pour l'identité), le mécanisme de B compté `(4, 0, 0, 4)`, le couple B/C universel sur
   les seuls axiomes standard, et la mise en regard bornée avec les Φ mesurés.

Ouverture : la **Phase 2** du lake — la cardinalité quantitative `|imA| · |imB| − |im f|`
comme définition formalisée, dont ce carnet a montré qu'elle se mesure par énumération dès
que `n` est concret ; et la question laissée au lecteur de l'exercice 3.

## 8. Exercices

Trois exercices, du calcul au débat. Les cellules s'exécutent telles quelles : compléter le
TODO puis re-exécuter.

In [10]:
-- Exercice 1 : le defaut de l'identite sur une AUTRE coupe
-- TODO etudiant : d'apres le theoreme A, predire le tuple
-- (|imA|, |imB|, |imA|*|imB|, |im f|) de l'identite sur cutB (cote A = {0,1}, cote B = {2}),
-- puis le defect cardinal. Ecrire la prediction en commentaire.
-- Etape 2 : decommenter la ligne de controle et verifier.
#eval "Exercice 1 a completer"
-- #eval ((partImageT 3 cutB (fun s => s)).length,
--        (partImageF 3 cutB (fun s => s)).length,
--        (partImageT 3 cutB (fun s => s)).length * (partImageF 3 cutB (fun s => s)).length,
--        (imageBits 3 (fun s => s)).length)

-- Exercice 1 : le defaut de l'identite sur une AUTRE coupe
-- TODO etudiant : d'apres le theoreme A, predire le tuple
-- (|imA|, |imB|, |imA|*|imB|, |im f|) de l'identite sur cutB (cote A = {0,1}, cote B = {2}),
-- puis le defect cardinal. Ecrire la prediction en commentaire.
-- Etape 2 : decommenter la ligne de controle et verifier.
#eval "Exercice 1 a completer"
─────▶  "Exercice 1 a completer"
-- #eval ((partImageT 3 cutB (fun s => s)).length,
--        (partImageF 3 cutB (fun s => s)).length,
--        (partImageT 3 cutB (fun s => s)).length * (partImageF 3 cutB (fun s => s)).length,
--        (imageBits 3 (fun s => s)).length)
--% env 9

Raw input:
{"cmd": "-- Exercice 1 : le defaut de l'identite sur une AUTRE coupe\n-- TODO etudiant : d'apres le theoreme A, predire le tuple\n-- (|imA|, |imB|, |imA|*|imB|, |im f|) de l'identite sur cutB (cote A = {0,1}, cote B = {2}),\n-- puis le defect cardinal. Ecrire la prediction en commentaire.\n-- Etape 2 : decommenter la ligne de controle et verifier.\n#eval \"Exercice 1 a completer\"\n-- #eval ((partImageT 3 cutB (fun s => s)).length,\n--        (partImageF 3 cutB (fun s => s)).length,\n--        (partImageT 3 cutB (fun s => s)).length * (partImageF 3 cutB (fun s => s)).length,\n--        (imageBits 3 (fun s => s)).length)", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 1 a completer\""}],
 "env": 9}

In [11]:
-- Exercice 2 : le systeme constant
-- TODO etudiant : le systeme fun _ => (fun _ => false) envoie tout etat sur l'etat tout-faux.
-- Predire son tuple (|imA|, |imB|, produit, |im f|) sur cutA, son defaut cardinal,
-- et son verdict (decomposable ou integrated). Repondre en commentaire, en citant
-- la regle : quand l'image est un singleton, que vaut le produit des images partielles ?
#eval "Exercice 2 a completer"


-- Exercice 2 : le systeme constant
-- TODO etudiant : le systeme fun _ => (fun _ => false) envoie tout etat sur l'etat tout-faux.
-- Predire son tuple (|imA|, |imB|, produit, |im f|) sur cutA, son defaut cardinal,
-- et son verdict (decomposable ou integrated). Repondre en commentaire, en citant
-- la regle : quand l'image est un singleton, que vaut le produit des images partielles ?
#eval "Exercice 2 a completer"
─────▶  "Exercice 2 a completer"

--% env 10

Raw input:
{"cmd": "-- Exercice 2 : le systeme constant\n-- TODO etudiant : le systeme fun _ => (fun _ => false) envoie tout etat sur l'etat tout-faux.\n-- Predire son tuple (|imA|, |imB|, produit, |im f|) sur cutA, son defaut cardinal,\n-- et son verdict (decomposable ou integrated). Repondre en commentaire, en citant\n-- la regle : quand l'image est un singleton, que vaut le produit des images partielles ?\n#eval \"Exercice 2 a completer\"\n", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 2 a completer\""}],
 "env": 10}

In [12]:
-- Exercice 3 : un integre a dynamique RICHE ?
-- TODO etudiant : la critique d'Aaronson ne dit pas que tout integre est trivial.
-- Proposer (en commentaire) la forme d'un systeme de State 3 qui serait integrated
-- au sens du proxy MAIS dont une orbite ne se stabiliserait pas en deux pas --
-- par exemple un cycle de longueur 3. Dire ce que le theoreme C a exactement
-- de specifique au broadcast, et pourquoi il ne s'applique pas a votre systeme.
#eval "Exercice 3 a completer"


-- Exercice 3 : un integre a dynamique RICHE ?
-- TODO etudiant : la critique d'Aaronson ne dit pas que tout integre est trivial.
-- Proposer (en commentaire) la forme d'un systeme de State 3 qui serait integrated
-- au sens du proxy MAIS dont une orbite ne se stabiliserait pas en deux pas --
-- par exemple un cycle de longueur 3. Dire ce que le theoreme C a exactement
-- de specifique au broadcast, et pourquoi il ne s'applique pas a votre systeme.
#eval "Exercice 3 a completer"
─────▶  "Exercice 3 a completer"

--% env 11

Raw input:
{"cmd": "-- Exercice 3 : un integre a dynamique RICHE ?\n-- TODO etudiant : la critique d'Aaronson ne dit pas que tout integre est trivial.\n-- Proposer (en commentaire) la forme d'un systeme de State 3 qui serait integrated\n-- au sens du proxy MAIS dont une orbite ne se stabiliserait pas en deux pas --\n-- par exemple un cycle de longueur 3. Dire ce que le theoreme C a exactement\n-- de specifique au broadcast, et pourquoi il ne s'applique pas a votre systeme.\n#eval \"Exercice 3 a completer\"\n", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "\"Exercice 3 a completer\""}],
 "env": 11}

***

*Capstone de l'escalier `iit_lean` — marche 3 sur 3. Le lake vit dans
`IIT/iit_lean/` (namespace `IIT`, sans dépendance Mathlib) ; la note de
conception de l'escalier est `docs/lean/iit-lean-escalier.md`. La moitié Python de
l'objet qui a mordu IIT est mesurée dans l'[IIT-06-L].*